 ### Example use of this workflow at the end ###

In [ ]:
# Reference / inspiration:
# Self-correcting code-generation agent pattern adapted from:
# https://learnopencv.com/langgraph-self-correcting-agent-code-generation/

In [ ]:
%pip install langgraph langchain-core langchain-huggingface requests beautifulsoup4

In [ ]:
"""
LangGraph Code-Generation Workflow
===================================
An autonomous workflow that:
  1. Asks the user for a HuggingFace model + code generation prompt
  2. Generates Python code with that model
  3. Executes the code, checks for errors
  4. On failure (with attempts remaining): auto-retries generation
  5. On success or exhausted attempts: asks the user whether to change
     the prompt, provide URLs for extra context, or end the workflow

Dependencies:
    pip install langgraph langchain-core langchain-huggingface requests beautifulsoup4
"""

In [3]:
import os
import re
import sys
import uuid
import tempfile
import subprocess
from typing import Annotated, Sequence, TypedDict, Optional

import requests
from bs4 import BeautifulSoup

from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage, AIMessage
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver


In [6]:
# Fill in your Hugging Face API token below (or set it as an environment
# variable HUGGINGFACEHUB_API_TOKEN before running this script).
HUGGINGFACEHUB_API_TOKEN = ""  # <-- put your key here

os.environ["HUGGINGFACEHUB_API_TOKEN"] = os.environ.get(
    "HUGGINGFACEHUB_API_TOKEN", HUGGINGFACEHUB_API_TOKEN
)

In [28]:
# ---------------------------------------------------------------------------
# Step 1: Agent State + HuggingFace API key setup
# ---------------------------------------------------------------------------
class CodeGenState(TypedDict):
    max_iterations: int
    iter: int
    messages: Annotated[Sequence[BaseMessage], add_messages]
    code: Optional[str]
    error: bool
    user_prompt: str
    model: str

# ---------------------------------------------------------------------------
# Step 2: enquiry node
# ---------------------------------------------------------------------------
CODE_ONLY_SYSTEM_PROMPT = (
    "You are an expert Python code generation assistant.\n"
    "STRICT OUTPUT RULES:\n"
    "1. Respond with EXECUTABLE PYTHON CODE ONLY.\n"
    "2. Do NOT include markdown code fences (no ```python or ```).\n"
    "3. Do NOT include any explanations, comments about your reasoning, "
    "or natural language text outside of the code.\n"
    "4. The code must be syntactically valid and runnable as-is in a "
    "standard Python 3.9+ environment using only common standard-library "
    "or widely available packages.\n"
    "5. If the user's request is ambiguous, make the most reasonable "
    "assumption and still output only code (use comments inside the code "
    "for any clarifying notes).\n"
    "6. If you are fixing a previous error, output the FULL corrected "
    "code, not just the diff or the changed lines."
)

def enquiry_node(state: CodeGenState) -> CodeGenState:
    """
    Entry node: collects the target HF model name and the user's code
    generation prompt, then seeds the message history with a strict
    system instruction that forces code-only output.
    """
    model_name = input(
        "Enter the HuggingFace model repo id to use for code generation "
        "(e.g. 'bigcode/starcoder2-15b'): "
    ).strip()

    user_prompt = input(
        "Describe the Python code you want generated: "
    ).strip()

    seed_messages = [
        SystemMessage(content=CODE_ONLY_SYSTEM_PROMPT),
        HumanMessage(content=f"Task: {user_prompt}\n\nGenerate the code now."),
    ]

    return {
        "max_iterations": state.get("max_iterations", 5),
        "iter": 0,
        "messages": seed_messages,
        "code": None,
        "error": False,
        "user_prompt": user_prompt,
        "model": model_name,
    }


# ---------------------------------------------------------------------------
# Step 3: code generation node
# ---------------------------------------------------------------------------
def _strip_code_fences(text: str) -> str:
    """
    Safety net: some HF models ignore the 'no markdown fences' instruction.
    Strip ```python ... ``` or ``` ... ``` wrappers if present.
    """
    text = text.strip()
    fence_pattern = r"^```(?:python)?\s*\n?(.*?)\n?```$"
    match = re.match(fence_pattern, text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return text


def generate_code_node(state: CodeGenState) -> CodeGenState:
    """
    Calls the HuggingFace-hosted LLM (specified by state['model']) with the
    accumulated message history and produces a new code candidate.
    """
    llm_endpoint = HuggingFaceEndpoint(
        repo_id=state["model"],
        task="text-generation",
        max_new_tokens=2048,
        temperature=0.2,
        provider="together",
        huggingfacehub_api_token=os.environ.get("HUGGINGFACEHUB_API_TOKEN"),
    )
    chat_model = ChatHuggingFace(llm=llm_endpoint)

    response = chat_model.invoke(state["messages"])
    raw_output = response.content

    generated_code = _strip_code_fences(raw_output)

    return {
        "iter": state["iter"] + 1,
        "code": generated_code,
        "messages": [AIMessage(content=generated_code)],
    }


# ---------------------------------------------------------------------------
# Step 4: code checking and executing node (with temp file cleanup)
# ---------------------------------------------------------------------------
def check_and_execute_node(state: CodeGenState) -> CodeGenState:
    """
    Attempts to execute the most recently generated code in an isolated
    subprocess. Reports progress to stdout, and on failure appends a
    HumanMessage asking the model to fix the error, including the traceback.
    Cleans up the temporary file regardless of outcome.
    """
    print(f"\n[Attempt {state['iter']}/{state['max_iterations']}] "
          f"Generating and validating code...")
    print("-" * 60)
    print("Generated code:\n")
    print(state["code"])
    print("-" * 60)
    print("Attempting execution...")

    tmp_path = None
    try:
        with tempfile.NamedTemporaryFile(
            mode="w", suffix=".py", delete=False, encoding="utf-8"
        ) as tmp_file:
            tmp_file.write(state["code"])
            tmp_path = tmp_file.name

        try:
            result = subprocess.run(
                [sys.executable, tmp_path],
                capture_output=True,
                text=True,
                timeout=30,
            )
        except subprocess.TimeoutExpired:
            print("Result: FAILED (execution timed out after 30s)")
            error_msg = (
                "Execution timed out after 30 seconds. "
                "Possible infinite loop or blocking call."
            )
            return {
                "error": True,
                "messages": [
                    HumanMessage(
                        content=f"Please fix the error and try again.\n\n"
                                f"Error:\n{error_msg}"
                    )
                ],
            }

        if result.returncode == 0:
            print("Result: SUCCESS — code executed without errors.")
            if result.stdout.strip():
                print(f"Program output:\n{result.stdout}")
            return {"error": False}
        else:
            print("Result: FAILED — code raised an error during execution.")
            print(f"Traceback:\n{result.stderr}")
            return {
                "error": True,
                "messages": [
                    HumanMessage(
                        content=f"Please fix the error and try again.\n\n"
                                f"Error:\n{result.stderr.strip()}"
                    )
                ],
            }
    finally:
        # Clean up the temp file regardless of success/failure/timeout
        if tmp_path and os.path.exists(tmp_path):
            try:
                os.unlink(tmp_path)
            except OSError as cleanup_err:
                print(f"Warning: failed to delete temp file {tmp_path}: {cleanup_err}")


# ---------------------------------------------------------------------------
# Step 5: prompt changing node
# ---------------------------------------------------------------------------
def prompt_change_node(state: CodeGenState) -> CodeGenState:
    """
    Allows the user to supply a new/revised prompt for code generation.
    Appends the new prompt to message history so the LLM sees it as an
    explicit instruction update, and resets iteration bookkeeping since
    this effectively starts a new generation goal.
    """
    print("\n--- Prompt Update ---")
    print(f"Current prompt: {state['user_prompt']}")

    new_prompt = input(
        "Enter a new prompt for code generation "
        "(leave blank to keep the current one): "
    ).strip()

    if not new_prompt:
        print("No change made. Keeping existing prompt.")
        return {}

    print(f"Prompt updated to: {new_prompt}")

    return {
        "user_prompt": new_prompt,
        "iter": 0,
        "error": False,
        "messages": [
            HumanMessage(
                content=f"Updated task: {new_prompt}\n\n"
                        f"Please generate new code for this updated task, "
                        f"following the same output rules as before "
                        f"(code only, no markdown fences)."
            )
        ],
    }


# ---------------------------------------------------------------------------
# Step 6: information scraping node
# ---------------------------------------------------------------------------
def _scrape_url(url: str, max_chars: int = 3000) -> str:
    """
    Fetches a URL and extracts visible text content, trimmed to max_chars
    to avoid blowing up the context window.
    """
    try:
        resp = requests.get(url, timeout=10, headers={"User-Agent": "Mozilla/5.0"})
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")

        for tag in soup(["script", "style", "noscript", "header", "footer", "nav"]):
            tag.decompose()

        text = " ".join(soup.get_text(separator=" ").split())
        return text[:max_chars]
    except Exception as e:
        return f"[Failed to scrape {url}: {e}]"


def info_scraping_node(state: CodeGenState) -> CodeGenState:
    """
    Prompts the user for up to 3 URLs to scrape for supplementary
    information (e.g., API docs, library references), and appends the
    scraped content to message history as additional context for the
    next code generation attempt.
    """
    print("\n--- Information Scraping ---")
    print("Enter up to 3 URLs to scrape for extra context "
          "(press Enter on a blank line to stop early).")

    urls = []
    for i in range(3):
        url = input(f"URL {i + 1}/3: ").strip()
        if not url:
            break
        urls.append(url)

    if not urls:
        print("No URLs provided. Skipping scrape.")
        return {}

    scraped_sections = []
    for url in urls:
        print(f"Scraping: {url} ...")
        content = _scrape_url(url)
        scraped_sections.append(f"Source: {url}\n{content}")

    combined_context = "\n\n---\n\n".join(scraped_sections)

    return {
        "messages": [
            HumanMessage(
                content=f"Here is additional reference information scraped "
                        f"from the following source(s). Use it to inform "
                        f"the code you generate, where relevant:\n\n"
                        f"{combined_context}"
            )
        ],
    }


# ---------------------------------------------------------------------------
# Step 7: routing function (single conditional edge function)
# ---------------------------------------------------------------------------
def route_after_check(state: CodeGenState) -> str:
    """
    Conditional edge function evaluated after check_and_execute_node.

    Returns one of: "ReGenerate", "ChangePrompt", "ScrapeInfo", "End"
    for use with add_conditional_edges().
    """
    if state["error"] and state["iter"] < state["max_iterations"]:
        return "ReGenerate"

    print("\n" + "=" * 60)
    if state["error"]:
        print(f"FAILED: exceeded max iterations "
              f"({state['iter']}/{state['max_iterations']}) "
              f"without producing working code.")
    else:
        print(f"SUCCESS: working code generated in "
              f"{state['iter']}/{state['max_iterations']} attempt(s).")
    print("=" * 60)

    print("\nWhat would you like to do next?")
    print("  1. Change the prompt and try again")
    print("  2. Provide URL(s) for extra context and try again")
    print("  3. End the workflow")

    choice = input("Enter 1, 2, or 3: ").strip()

    mapping = {"1": "ChangePrompt", "2": "ScrapeInfo", "3": "End"}
    action = mapping.get(choice)

    if action is None:
        print("Unrecognized input — ending workflow by default.")
        action = "End"

    return action


In [29]:
# ---------------------------------------------------------------------------
# Step 8: graph construction
# ---------------------------------------------------------------------------
workflow = StateGraph(CodeGenState)

workflow.add_node("Enquiry", enquiry_node)
workflow.add_node("CodeGeneration", generate_code_node)
workflow.add_node("CodeExecution&ErrorReport", check_and_execute_node)
workflow.add_node("ChangePrompt", prompt_change_node)
workflow.add_node("ScrapeInfo", info_scraping_node)

workflow.add_edge(START, "Enquiry")
workflow.add_edge("Enquiry", "CodeGeneration")
workflow.add_edge("CodeGeneration", "CodeExecution&ErrorReport")

workflow.add_conditional_edges(
    "CodeExecution&ErrorReport",
    route_after_check,
    {
        "ReGenerate": "CodeGeneration",
        "ChangePrompt": "ChangePrompt",
        "ScrapeInfo": "ScrapeInfo",
        "End": END,
    },
)

workflow.add_edge("ChangePrompt", "CodeGeneration")
workflow.add_edge("ScrapeInfo", "CodeGeneration")

memory = MemorySaver()
app = workflow.compile(checkpointer=memory)

# Draw the graph
try:
    graph_png = app.get_graph().draw_mermaid_png()
    with open("codegen_workflow_graph.png", "wb") as f:
        f.write(graph_png)
    print("Graph diagram saved to codegen_workflow_graph.png")
except Exception as e:
    print(f"Could not render PNG ({e}). Falling back to Mermaid source:\n")
    print(app.get_graph().draw_mermaid())

Graph diagram saved to codegen_workflow_graph.png


In [ ]:
# ---------------------------------------------------------------------------
# Step 9: initial state + graph invocation
# ---------------------------------------------------------------------------
def run_workflow(max_iterations: int = 5):
    """
    Sets up the initial state and invokes the compiled LangGraph workflow.
    A fresh thread_id is generated per run so MemorySaver checkpoints this
    session distinctly from any prior runs.
    """
    initial_state: CodeGenState = {
        "max_iterations": max_iterations,
        "iter": 0,
        "messages": [],
        "code": None,
        "error": False,
        "user_prompt": "",
        "model": "",
    }

    thread_id = str(uuid.uuid4())
    config = {
        "configurable": {"thread_id": thread_id},
        "recursion_limit": max(50, max_iterations * 4),
    }

    print(f"Starting workflow (thread_id={thread_id})...\n")

    final_state = app.invoke(initial_state, config=config)

    print("\n" + "=" * 60)
    print("WORKFLOW COMPLETE")
    print("=" * 60)
    print(f"Final iteration count: {final_state['iter']}/{final_state['max_iterations']}")
    print(f"Final error status: {final_state['error']}")
    print("\nFinal code:\n")
    print(final_state["code"] if final_state["code"] else "[No code generated]")

    return final_state


if __name__ == "__main__":
    run_workflow(max_iterations=5)


### Example run: results & notes

**Model used:** `deepseek-ai/DeepSeek-V4-Pro` (via Hugging Face Inference Providers)

**Prompt given:** *"Collect the air quality observation stations' position in Guangdong
Province, China and map them out"*

In [ ]:
import os
import requests
import folium
from folium import plugins

# Try to get WAQI API token from environment variable
WAQI_TOKEN = os.environ.get('WAQI_TOKEN', '')

# Guangdong approximate bounding box
LAT_MIN, LAT_MAX = 20.13, 25.31
LON_MIN, LON_MAX = 109.65, 117.20

def fetch_stations_from_waqi():
    """Fetch air quality stations from WAQI API for Guangdong region."""
    if not WAQI_TOKEN:
        raise ValueError("WAQI_TOKEN not set")
    url = f"https://api.waqi.info/map/bounds?latlng={LAT_MIN},{LON_MIN},{LAT_MAX},{LON_MAX}&token={WAQI_TOKEN}"
    resp = requests.get(url, timeout=10)
    resp.raise_for_status()
    data = resp.json()
    if data['status'] != 'ok':
        raise RuntimeError(f"WAQI API error: {data.get('data', 'unknown')}")
    stations = []
    for item in data['data']:
        if 'station' in item:
            station = item['station']
            stations.append({
                'name': station.get('name', 'Unknown'),
                'lat': station['geo'][0],
                'lon': station['geo'][1],
                'aqi': station.get('aqi', 'N/A')
            })
    return stations

def get_hardcoded_stations():
    """Fallback: known air quality stations in Guangdong (approximate)."""
    return [
        {'name': 'Guangzhou', 'lat': 23.1291, 'lon': 113.2644, 'aqi': 'N/A'},
        {'name': 'Shenzhen', 'lat': 22.5431, 'lon': 114.0579, 'aqi': 'N/A'},
        {'name': 'Zhuhai', 'lat': 22.2707, 'lon': 113.5767, 'aqi': 'N/A'},
        {'name': 'Foshan', 'lat': 23.0218, 'lon': 113.1214, 'aqi': 'N/A'},
        {'name': 'Dongguan', 'lat': 23.0207, 'lon': 113.7518, 'aqi': 'N/A'},
        {'name': 'Zhongshan', 'lat': 22.5159, 'lon': 113.3926, 'aqi': 'N/A'},
        {'name': 'Jiangmen', 'lat': 22.5787, 'lon': 113.0819, 'aqi': 'N/A'},
        {'name': 'Huizhou', 'lat': 23.1118, 'lon': 114.4158, 'aqi': 'N/A'},
        {'name': 'Zhaoqing', 'lat': 23.0471, 'lon': 112.4650, 'aqi': 'N/A'},
        {'name': 'Shantou', 'lat': 23.3535, 'lon': 116.6819, 'aqi': 'N/A'},
        {'name': 'Zhanjiang', 'lat': 21.2707, 'lon': 110.3594, 'aqi': 'N/A'},
        {'name': 'Maoming', 'lat': 21.6632, 'lon': 110.9254, 'aqi': 'N/A'},
        {'name': 'Shaoguan', 'lat': 24.8104, 'lon': 113.5976, 'aqi': 'N/A'},
        {'name': 'Meizhou', 'lat': 24.2886, 'lon': 116.1225, 'aqi': 'N/A'},
        {'name': 'Shanwei', 'lat': 22.7866, 'lon': 115.3753, 'aqi': 'N/A'},
        {'name': 'Heyuan', 'lat': 23.7437, 'lon': 114.7004, 'aqi': 'N/A'},
        {'name': 'Yangjiang', 'lat': 21.8582, 'lon': 111.9826, 'aqi': 'N/A'},
        {'name': 'Qingyuan', 'lat': 23.6822, 'lon': 113.0560, 'aqi': 'N/A'},
        {'name': 'Chaozhou', 'lat': 23.6568, 'lon': 116.6226, 'aqi': 'N/A'},
        {'name': 'Jieyang', 'lat': 23.5497, 'lon': 116.3727, 'aqi': 'N/A'},
        {'name': 'Yunfu', 'lat': 22.9151, 'lon': 112.0445, 'aqi': 'N/A'},
    ]

def main():
    stations = []
    try:
        stations = fetch_stations_from_waqi()
        print(f"Fetched {len(stations)} stations from WAQI API.")
    except Exception as e:
        print(f"Could not fetch from WAQI: {e}")
        print("Using hardcoded station list.")
        stations = get_hardcoded_stations()

    if not stations:
        print("No stations to map.")
        return

    # Create map centered on Guangdong
    map_center = [(LAT_MIN + LAT_MAX) / 2, (LON_MIN + LON_MAX) / 2]
    """change tiles='OpenStreetMap' to tiles='CartoDB positron' to fix 403 access denied """
    m = folium.Map(location=map_center, zoom_start=7, tiles='CartoDB positron')

    # Add markers
    for s in stations:
        popup_text = f"{s['name']}<br>AQI: {s['aqi']}"
        folium.Marker(
            location=[s['lat'], s['lon']],
            popup=popup_text,
            icon=folium.Icon(color='blue', icon='cloud')
        ).add_to(m)

    # Add fullscreen button
    plugins.Fullscreen().add_to(m)

    output_file = 'guangdong_air_quality_stations.html'
    m.save(output_file)
    print(f"Map saved to {output_file}")

if __name__ == '__main__':
    main()